In [ ]:
# Google Drive mount
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2: ZIP file extract
import zipfile
import os

zip_path = '/content/drive/MyDrive/archive(1).zip'
extract_dir = '/content/dataset'

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print(" Dataset extract!")
else:
    print(" ZIP file missing. Check path are correct?")


extracted = os.listdir(extract_dir)
print("Extracted items:", extracted)


if len(extracted) == 1 and os.path.isdir(os.path.join(extract_dir, extracted[0])):
    data_dir = os.path.join(extract_dir, extracted[0])
else:
    data_dir = extract_dir

print("✅ Data directory:", data_dir)

In [ ]:
# Cell 3: Required libraries install
!pip install gradio --quiet

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import numpy as np
import gradio as gr
from PIL import Image
import os

In [ ]:
# Cell 4: Global variables
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
model_path = 'plant_disease_model.h5'
class_names = None

In [ ]:
# Cell 5: Check model in Drive
model_path_drive = '/content/drive/MyDrive/plant_disease_model.h5'   # Drive path
model_path_local = 'plant_disease_model.h5'                         # local path

if os.path.exists(model_path_drive):
    print("✅ Model Exist in Drive Loading...")
    model = tf.keras.models.load_model(model_path_drive)
    print("✅ Model Are Load From Drive")

    # Class names load
    if os.path.exists('/content/drive/MyDrive/class_names.txt'):
        with open('/content/drive/MyDrive/class_names.txt', 'r') as f:
            class_names = [line.strip() for line in f.readlines()]
        print("✅ Class names load from Drive")
    else:
        # If class_names.txt are not exist generate from datase
        print("class_names.txt are not found, generate from dataset")
        from tensorflow.keras.preprocessing.image import ImageDataGenerator
        dummy_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
            data_dir,
            target_size=IMG_SIZE,
            batch_size=1,
            class_mode='categorical'
        )
        class_names = list(dummy_gen.class_indices.keys())
        print(f"✅ {len(class_names)} class are names load.")

    train_model = False   #skip Training

elif os.path.exists(model_path_local):
    print("✅ Model are found in local")
    model = tf.keras.models.load_model(model_path_local)
    print("✅ Model are load from local")
    # ... same class name loading logic ...
    train_model = False
else:
    print(" Model are not found anywhere.have to Training.")
    train_model = True

In [ ]:
# Cell 6: Data Generators (Run Only If you Train Model)
if 'train_model' in locals() and train_model:
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        validation_split=0.2
    )

    train_generator = train_datagen.flow_from_directory(
        data_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='training'
    )

    val_generator = train_datagen.flow_from_directory(
        data_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='validation'
    )

    num_classes = train_generator.num_classes
    class_names = list(train_generator.class_indices.keys())
    print(f"Total classes: {num_classes}")
    print("Sample class names:", class_names[:5])
else:
    print("Data generators ki zaroorat nahi, model already load hai.")

In [ ]:
# Cell 7: Model build and train
if 'train_model' in locals() and train_model:
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
        layers.MaxPooling2D(2,2),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),
        layers.Conv2D(128, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    model.summary()

    # Callbacks
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(model_path, monitor='val_accuracy', save_best_only=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3)
    ]

    # Train
    history = model.fit(
        train_generator,
        steps_per_epoch=train_generator.samples // BATCH_SIZE,
        epochs=15,
        validation_data=val_generator,
        validation_steps=val_generator.samples // BATCH_SIZE,
        callbacks=callbacks
    )

    model.save(model_path)
    print(" Model train and saved!")

    # Plot graph
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(history.history['accuracy'], label='Train Acc')
    plt.plot(history.history['val_accuracy'], label='Val Acc')
    plt.title('Accuracy')
    plt.legend()
    plt.subplot(1,2,2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title('Loss')
    plt.legend()
    plt.show()
else:
    print("Training skip (model already exist).")

In [ ]:
# Cell 8: Prediction functions (modified with threshold)
def preprocess_image(img):
    img = img.resize(IMG_SIZE)
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    return img_array

def predict(img):
    img_array = preprocess_image(img)
    pred = model.predict(img_array)
    idx = np.argmax(pred[0])
    pred_class = class_names[idx]
    confidence = pred[0][idx] * 100
    return pred_class, confidence

def predict_and_display(img):
    img_array = preprocess_image(img)
    pred = model.predict(img_array)
    idx = np.argmax(pred[0])
    confidence = pred[0][idx] * 100
    predicted_class = class_names[idx]

    if confidence < 60:
        return f"🌿 **Result:** Healthy Leaf (Probably)\n🎯 Confidence: {confidence:.1f}%"
    else:
        if "healthy" in predicted_class.lower():
            return f"🌿 **Result:** {predicted_class} (Healthy Leaf)\n🎯 Confidence: {confidence:.1f}%"
        else:
            return f"🌿 **Disease:** {predicted_class}\n⚠️ Confidence: {confidence:.1f}%"

In [ ]:
# Cell 9: Gradio interface banayein jisme "Image" input me source options honge
demo = gr.Interface(
    fn=predict_and_display,
    inputs=gr.Image(type="pil", label="📸 Option: Upload ya Webcam", sources=["upload", "webcam"]),
    outputs=gr.Textbox(label="Prediction Result", lines=2),
    title="🌾 Plant Disease Detection System",
    description="""**Dono methods available hain:**
    1. **Upload** - apni machine se picture select karo
    2. **Webcam** - camera se capture karo (mobile/laptop mein allow karna)

    Model trained on multiple crops (Wheat, Tomato, Corn, Potato, Rice, etc.) with 47 disease classes.""",
    allow_flagging="never"
)

# Launch karo
demo.launch(debug=True, share=True)